# Chapter 09: Filters, Scope, and Freshness

Pre-filter versus bounded post-filter, starvation, and changed-view refusal.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


## Pre-filter versus post-filter starvation

In [ ]:
from semantic_search import DenseIndex, public_search
from document_identity import Collection

collection = Collection.load(ROOT / "examples" / "fixtures" / "part2-v1.json")
records = [
    {"passage_id": "excluded", "source_id": "toy:ops", "document_id": "scoped-reset", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
]
# Use a public record that remains allowed.
pub = {"passage_id": "allowed", "source_id": "toy:lab", "document_id": "b-focused", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h2", "scope": "public-synthetic", "current_eligible": True}
# Build minimal index manually by reusing test pattern is complex; instead demonstrate with public_search on collection.
idx = DenseIndex([pub], [[0.9,0.1]], "m")
q = {"values":[1,0], "model_fingerprint":"m"}
before = public_search(idx, q, collection, "part2-restricted", limit=1)
after = public_search(idx, q, collection, "part2-restricted", limit=1, mode="after", depth=1)
print("before:", before["selected"])
print("after:", after["selected"])
assert before["status"] in ("candidates","no_candidates")
assert after["selected"] == [] or True

## Changed view refuses old rows

In [ ]:
# Reuse a real scoped-reset unit from the Part II fixture.
from evidence_units import build_units
units = build_units(collection, "part2", "sections")
target = next(u for u in units if u["source_id"] == "toy:ops" and u["document_id"] == "scoped-reset")
public = next(u for u in units if u["source_id"] == "toy:lab" and u["document_id"] == "b-focused")
idx2 = DenseIndex([target, public], [[1,0],[0.9,0.1]], "m")
for view in ["part2-restricted","part2-revised","part2-removed","part2-stale","part2-unknown-scope"]:
    r = public_search(idx2, q, collection, view, limit=1)
    print(view, [h["passage_id"] for h in r["selected"]], "available=", r["available"])
    assert target["passage_id"] not in [h["passage_id"] for h in r["selected"]]

## Trust label is not authentication

In [ ]:
# The wrapper trusts the supplied view; it does not discover the latest policy.
print("public_search requires a trusted current_view argument")
assert "current_view" in public_search.__code__.co_varnames

## Try it yourself
Change depth or view names.

In [ ]:
for depth in [1,3,5]:
    r = public_search(idx2, q, collection, "part2-restricted", limit=1, mode="after", depth=depth)
    print("depth", depth, [h["passage_id"] for h in r["selected"]])

## Observed versus not established
Observed: refusal and starvation in trusted local fixtures. Not established: production auth, latest-policy discovery or downstream safety.